In [2]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

/kaggle/input/project-multi/test_random_400-2.csv
/kaggle/input/project-multi/test_top_cosine_200-2.csv
/kaggle/input/project-multi/test_top_rougeL_200-2.csv
/kaggle/input/project-multi/train_data.csv
/kaggle/input/project-multi/test_random_600-2.csv


In [3]:
!pip install -U "transformers==4.44.2" "accelerate==0.34.2" "peft==0.11.1" optuna evaluate rouge_score

huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.7/43.7 kB 1.9 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.5/9.5 MB 86.9 MB/s eta 0:00:00:00:010:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 324.4/324.4 kB 22.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 251.6/251.6 kB 15.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 404.7/404.7 kB 24.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 5.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 193.6/193.6 kB 12.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 82.4 MB/s eta 0:00:00:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.4/363.4 MB 4.6 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.8/13.8 MB 91.2 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.6/24.6 MB 73.3 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

In [7]:
from huggingface_hub import login

login("your_token_hera")

# Percentage of exceeding token length

In [7]:
import pandas as pd
from transformers import AutoTokenizer
from tqdm import tqdm

# ===== CONFIG =====
CSV_PATH = "/kaggle/input/project-multi/test_random_600-2.csv"
TEXT_COL = "description_html_clean"
MODEL = "google/flan-t5-xl"   # đổi sang t5-large, facebook/bart-large, gemma, llama…
MAX_SOURCE_LEN = 512

# ===== LOAD DATA =====
df = pd.read_csv(CSV_PATH)
df = df[df[TEXT_COL].notna()]   # bỏ NaN
texts = df[TEXT_COL].tolist()

# ===== LOAD TOKENIZER =====
tokenizer = AutoTokenizer.from_pretrained(MODEL)

token_lengths = []
overflow_count = 0

for txt in tqdm(texts, desc="Tokenizing"):
    toks = tokenizer.encode(txt, add_special_tokens=True)
    L = len(toks)
    token_lengths.append(L)
    if L > MAX_SOURCE_LEN:
        overflow_count += 1

total = len(token_lengths)
pct_overflow = overflow_count / total * 100
avg_len = sum(token_lengths) / total

print(f"Model tokenizer: {MODEL}")
print(f"Total samples: {total}")
print(f"Average token length: {avg_len:.2f}")
print(f"Max token length: {max(token_lengths)}")
print(f"Min token length: {min(token_lengths)}")
print(f"% samples > {MAX_SOURCE_LEN} tokens: {pct_overflow:.2f}%")

tokenizer_config.json: 0.00B [00:00, ?B/s]

spiece.model:   0%|          | 0.00/792k [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json: 0.00B [00:00, ?B/s]

/usr/local/lib/python3.11/dist-packages/transformers/tokenization_utils_base.py:1601: FutureWarning: `clean_up_tokenization_spaces` was not set. It will be set to `True` by default. This behavior will be depracted in transformers v4.45, and will be then set to `False` by default. For more details check this issue: https://github.com/huggingface/transformers/issues/31884
  warnings.warn(
Tokenizing: 100%|██████████| 600/600 [00:00<00:00, 1846.13it/s]

Model tokenizer: google/flan-t5-xl
Total samples: 600
Average token length: 148.75
Max token length: 499
Min token length: 8
% samples > 512 tokens: 0.00%


# T5

In [8]:
# infer_t5_app.py
# Inference-only với google-t5/t5-3b
# - Đọc 2 file CSV test (400 & 600)
# - Mỗi file chạy 3 lần inference để log token/time
# - Xuất CSV dự đoán + JSONL log cho từng lần
# - Không cắt/regex, decode nguyên văn

import os, time, json
import pandas as pd
import torch
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

# ========= CONFIG =========
MODEL_NAME     = "google/flan-t5-xl"
INPUT_FILES    = {
    "test400": "/kaggle/input/project-multi/test_random_400-2.csv",
    "test600": "/kaggle/input/project-multi/test_random_600-2.csv",
}
INPUT_COL      = "description_html_clean"   # đổi cho đúng cột của bạn
OUTPUT_DIR     = "./t5_infer"
MAX_SOURCE_LEN = 512

GEN_KWARGS = dict(
    max_new_tokens=64,      # target ~17 tokens, dư địa an toàn
    num_beams=4,
    do_sample=False,
    no_repeat_ngram_size=3,
    length_penalty=1.0,
    early_stopping=True,
)

# ========= PROMPT (tùy chọn cho use-case app) =========
USE_APP_PROMPT = False
def build_prompt_for_app(html_text: str) -> str:
    return (
        "summarize: You are an expert app store editor. "
        "Given the following app description in HTML format, summarize it in 2-3 sentences, "
        "with a concise, engaging short description (max 80 characters) suitable for an app store listing. "
        f"App Description HTML:\n{html_text}\n"
        "Format your response as:\n"
        "Short Description: <your short description>\n\n"
    )

# ========= LOAD MODEL =========
print("⏳ Loading model...")
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, use_fast=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForSeq2SeqLM.from_pretrained(
    MODEL_NAME,
    torch_dtype=torch.float16 if torch.cuda.is_available() else torch.float32,
    device_map="auto",
)
model.eval()
device = "cuda" if torch.cuda.is_available() else "cpu"
print("✅ Model ready.")
# ========= LOOP FILE ĐAN XEN =========
os.makedirs(OUTPUT_DIR, exist_ok=True)

# Lịch chạy xen kẽ: mỗi file 3 lần
RUN_SCHEDULE = [  "test400", "test600","test400","test600", "test400"]
run_counters = {k: 0 for k in INPUT_FILES.keys()}

for tag in RUN_SCHEDULE:
    run_counters[tag] += 1
    run_id = run_counters[tag]

    file_path = INPUT_FILES[tag]
    assert os.path.exists(file_path), f"Missing: {file_path}"
    df = pd.read_csv(file_path)
    assert INPUT_COL in df.columns, f"Missing column: {INPUT_COL}"

    print(f"▶️ Running {tag}, round {run_id} ...")
    pred_rows = []
    infer_log_path = os.path.join(OUTPUT_DIR, f"log_{tag}_run{run_id}.jsonl")
    pred_path      = os.path.join(OUTPUT_DIR, f"pred_{tag}_run{run_id}.csv")

    with open(infer_log_path, "w", encoding="utf-8") as f_log:
        for i, row in df.iterrows():
            raw_src = str(row[INPUT_COL])
            src_text = build_prompt_for_app(raw_src) if USE_APP_PROMPT else raw_src

            # Tokenize
            inputs = tokenizer(
                src_text,
                return_tensors="pt",
                truncation=True,
                max_length=MAX_SOURCE_LEN,
            )
            inputs = {k: v.to(device) for k, v in inputs.items()}

            # Đếm input tokens
            input_tokens = int(inputs["attention_mask"].sum().item())

            # Generate + đo thời gian
            t0 = time.time()
            with torch.inference_mode():
                outputs = model.generate(**inputs, **GEN_KWARGS)
            latency = time.time() - t0

            # Decode
            decoded = tokenizer.decode(outputs[0], skip_special_tokens=True)

            # Đếm output tokens
            gen_tokens = int((outputs[0] != tokenizer.pad_token_id).sum().item())

            # Log JSONL
            f_log.write(json.dumps({
                "index": i,
                "input_tokens": input_tokens,
                "gen_tokens": gen_tokens,
                "latency_sec": round(latency, 4),
            }, ensure_ascii=False) + "\n")

            pred_rows.append({"index": i, "prediction": decoded})

    pd.DataFrame(pred_rows).to_csv(pred_path, index=False, encoding="utf-8")
    print(f"✅ Done {tag} run {run_id}: wrote {len(pred_rows)} rows → {pred_path}")


⏳ Loading model...


Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

✅ Model ready.
▶️ Running test400, round 1 ...
✅ Done test400 run 1: wrote 400 rows → ./t5_infer/pred_test400_run1.csv
▶️ Running test600, round 1 ...
✅ Done test600 run 1: wrote 600 rows → ./t5_infer/pred_test600_run1.csv
▶️ Running test400, round 2 ...
✅ Done test400 run 2: wrote 400 rows → ./t5_infer/pred_test400_run2.csv
▶️ Running test600, round 2 ...
✅ Done test600 run 2: wrote 600 rows → ./t5_infer/pred_test600_run2.csv
▶️ Running test400, round 3 ...
✅ Done test400 run 3: wrote 400 rows → ./t5_infer/pred_test400_run3.csv


# Peagsus xsum

In [3]:
# infer_pegasus_xsum.py
# Inference-only với google/pegasus-xsum
# - Đọc CSV, lấy cột INPUT_COL làm nguồn tóm tắt
# - Xuất CSV dự đoán + JSONL log token/time
# - Không cắt/regex, trả nguyên văn decode

import os, time, json
import pandas as pd
import torch
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

# ========= CONFIG =========
MODEL_NAME    = "google/pegasus-large"
INPUT_CSV     = "/kaggle/input/project-multi/test_random_400-2.csv"  # đổi nếu cần
INPUT_COL     = "description_html_clean"   
OUTPUT_DIR    = "./pegasus_xsum_infer"
PRED_CSV      = "pegasus_xsum_predictions.csv"
INFER_LOG     = "inference_token_time.jsonl"
USE_APP_PROMPT = False     # True: dùng prompt tóm tắt app từ HTML; False: tóm tắt trực tiếp văn bản
MAX_SOURCE_LEN = 512

# Tham số decode (có thể chỉnh)
GEN_KWARGS = dict(
    max_new_tokens=64,
    num_beams=4,
    do_sample=False,
    no_repeat_ngram_size=3,
    length_penalty=1.0,
    early_stopping=True,
)

# ========= PROMPT (tuỳ chọn cho use-case app) =========
def build_prompt_for_app(html_text: str) -> str:
    return (
        "You are an expert app store editor. "
        "Given the following app description in HTML format, summarize it in 2-3 sentences, "
        "with a concise, engaging short description (max 80 characters) suitable for an app store listing. "
        f"App Description HTML:\n{html_text}\n"
        "Format your response as:\n"
        "Short Description: <your short description>\n\n"
    )

# ========= LOAD DATA =========
assert os.path.exists(INPUT_CSV), f"Missing: {INPUT_CSV}"
df = pd.read_csv(INPUT_CSV)
assert INPUT_COL in df.columns, f"Missing column: {INPUT_COL}"

os.makedirs(OUTPUT_DIR, exist_ok=True)
infer_log_path = os.path.join(OUTPUT_DIR, INFER_LOG)
pred_path      = os.path.join(OUTPUT_DIR, PRED_CSV)

# ========= LOAD MODEL =========
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, use_fast=True)
model = AutoModelForSeq2SeqLM.from_pretrained(MODEL_NAME)
model.eval()
device = "cuda" if torch.cuda.is_available() else "cpu"
model.to(device)

# ========= INFERENCE =========
pred_rows = []
with open(infer_log_path, "w", encoding="utf-8") as f_log:
    for i, row in df.iterrows():
        raw_src = str(row[INPUT_COL])

        # Chọn nguồn tóm tắt
        src_text = build_prompt_for_app(raw_src) if USE_APP_PROMPT else raw_src

        # Tokenize
        inputs = tokenizer(
            src_text,
            return_tensors="pt",
            truncation=True,
            max_length=MAX_SOURCE_LEN,
        )
        inputs = {k: v.to(device) for k, v in inputs.items()}

        # Đếm input tokens
        input_tokens = int(inputs["attention_mask"].sum().item()) if "attention_mask" in inputs else int((inputs["input_ids"] != tokenizer.pad_token_id).sum().item())

        # Generate + đo thời gian
        t0 = time.time()
        with torch.no_grad():
            out = model.generate(**inputs, **GEN_KWARGS)
        latency = time.time() - t0

        # Decode FULL (không cắt, không regex)
        text = tokenizer.decode(out[0], skip_special_tokens=True, clean_up_tokenization_spaces=True)

        # Đếm output tokens (gần đúng, bỏ pad)
        gen_ids = out[0]
        try:
            output_tokens = int((gen_ids != tokenizer.pad_token_id).sum().item())
        except Exception:
            output_tokens = int(len(gen_ids))
        total_tokens = input_tokens + output_tokens
        tps = total_tokens / latency if latency > 0 else None

        # Lưu prediction
        pred_rows.append({"index": i, "prediction": text})

        # Log JSONL
        f_log.write(json.dumps({
            "event": "inference",
            "row_index": int(i),
            "input_tokens": int(input_tokens),
            "output_tokens": int(output_tokens),
            "total_tokens": int(total_tokens),
            "latency_sec": float(latency),
            "tokens_per_sec": float(tps) if tps is not None else None,
            "gen_kwargs": GEN_KWARGS,
            "timestamp": time.time(),
        }) + "\n")

# ========= SAVE PREDICTIONS =========
pd.DataFrame(pred_rows).to_csv(pred_path, index=False, encoding="utf-8")
print("Saved predictions to:", pred_path)
print("Inference token/time logs:", infer_log_path)

tokenizer_config.json:   0%|          | 0.00/88.0 [00:00<?, ?B/s]

config.json: 0.00B [00:00, ?B/s]

spiece.model:   0%|          | 0.00/1.91M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/65.0 [00:00<?, ?B/s]

/usr/local/lib/python3.11/dist-packages/transformers/tokenization_utils_base.py:1601: FutureWarning: `clean_up_tokenization_spaces` was not set. It will be set to `True` by default. This behavior will be depracted in transformers v4.45, and will be then set to `False` by default. For more details check this issue: https://github.com/huggingface/transformers/issues/31884
  warnings.warn(


pytorch_model.bin:   0%|          | 0.00/2.28G [00:00<?, ?B/s]

Some weights of PegasusForConditionalGeneration were not initialized from the model checkpoint at google/pegasus-large and are newly initialized: ['model.decoder.embed_positions.weight', 'model.encoder.embed_positions.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


generation_config.json:   0%|          | 0.00/260 [00:00<?, ?B/s]

Saved predictions to: ./pegasus_xsum_infer/pegasus_xsum_predictions.csv
Inference token/time logs: ./pegasus_xsum_infer/inference_token_time.jsonl


In [7]:
# infer_pegasus_schedule.py
# Inference-only với 2 model Pegasus (xsum & large-xsum)
# - Chạy xen kẽ file test: 600 → 400 → 600 → 400 → 600
# - Xuất CSV dự đoán + JSONL log token/time
# - Không regex, giữ nguyên văn bản decode

import os, time, json, gc
import pandas as pd
import torch
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

# ========= CONFIG =========
MODELS = {
    #"pegasus-large":       "google/pegasus-large",
    "pegasus-xsum": "google/pegasus-xsum",
}
INPUT_FILES = {
    "test400": "/kaggle/input/project-multi/test_random_400-2.csv",
    "test600": "/kaggle/input/project-multi/test_random_600-2.csv",
}
RUN_SCHEDULE = ["test400","test400"]

INPUT_COL      = "description_html_clean"
OUTPUT_ROOT    = "./pegasus_infer_schedule"
MAX_SOURCE_LEN = 512

GEN_KWARGS = dict(
    max_new_tokens=64,
    num_beams=4,
    do_sample=False,
    no_repeat_ngram_size=3,
    length_penalty=1.0,
    early_stopping=True,
)

USE_APP_PROMPT = False
def build_prompt_for_app(html_text: str) -> str:
    return (
        "You are an expert app store editor. "
        "Given the following app description in HTML format, summarize it in 2-3 sentences, "
        "with a concise, engaging short description (max 80 characters) suitable for an app store listing. "
        f"App Description HTML:\n{html_text}\n"
        "Format your response as:\n"
        "Short Description: <your short description>\n\n"
    )

# ========= MAIN LOOP =========
os.makedirs(OUTPUT_ROOT, exist_ok=True)
device = "cuda" if torch.cuda.is_available() else "cpu"

for model_tag, model_name in MODELS.items():
    print(f"\n==============================")
    print(f" Loading model: {model_name} ({model_tag})")
    print(f"==============================")

    OUT_DIR = os.path.join(OUTPUT_ROOT, model_tag)
    os.makedirs(OUT_DIR, exist_ok=True)

    tokenizer = AutoTokenizer.from_pretrained(model_name, use_fast=True)
    model = AutoModelForSeq2SeqLM.from_pretrained(model_name)
    model.eval().to(device)
    print("✅ Model ready.")

    run_counters = {"test400": 0, "test600": 0}

    for tag in RUN_SCHEDULE:
        run_counters[tag] += 1
        run_id = run_counters[tag]

        file_path = INPUT_FILES[tag]
        assert os.path.exists(file_path), f"Missing: {file_path}"
        df = pd.read_csv(file_path)
        assert INPUT_COL in df.columns, f"Missing column: {INPUT_COL}"

        print(f"▶️ [{model_tag}] Running {tag}, round {run_id} ...")
        pred_rows = []
        infer_log_path = os.path.join(OUT_DIR, f"log_{tag}_run{run_id}.jsonl")
        pred_path      = os.path.join(OUT_DIR, f"pred_{tag}_run{run_id}.csv")

        with open(infer_log_path, "w", encoding="utf-8") as f_log:
            for i, row in df.iterrows():
                raw_src = str(row[INPUT_COL])
                src_text = build_prompt_for_app(raw_src) if USE_APP_PROMPT else raw_src

                # Tokenize
                inputs = tokenizer(
                    src_text,
                    return_tensors="pt",
                    truncation=True,
                    max_length=MAX_SOURCE_LEN,
                )
                inputs = {k: v.to(device) for k, v in inputs.items()}
                input_tokens = int(inputs["attention_mask"].sum().item())

                # Generate
                t0 = time.time()
                with torch.no_grad():
                    out = model.generate(**inputs, **GEN_KWARGS)
                latency = time.time() - t0

                # Decode (nguyên văn)
                text = tokenizer.decode(out[0], skip_special_tokens=True, clean_up_tokenization_spaces=True)

                # Đếm output tokens
                gen_ids = out[0]
                try:
                    output_tokens = int((gen_ids != tokenizer.pad_token_id).sum().item())
                except Exception:
                    output_tokens = int(len(gen_ids))
                total_tokens = input_tokens + output_tokens
                tps = total_tokens / latency if latency > 0 else None

                pred_rows.append({"index": i, "prediction": text})

                f_log.write(json.dumps({
                    "event": "inference",
                    "row_index": int(i),
                    "input_tokens": input_tokens,
                    "output_tokens": output_tokens,
                    "total_tokens": total_tokens,
                    "latency_sec": float(latency),
                    "tokens_per_sec": float(tps) if tps else None,
                    "gen_kwargs": GEN_KWARGS,
                    "timestamp": time.time(),
                }) + "\n")

        pd.DataFrame(pred_rows).to_csv(pred_path, index=False, encoding="utf-8")
        print(f"✅ [{model_tag}] Done {tag} run {run_id}: {len(pred_rows)} rows → {pred_path}")

    # giải phóng RAM/GPU
    del model
    del tokenizer
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()


 Loading model: google/pegasus-xsum (pegasus-xsum)


Some weights of PegasusForConditionalGeneration were not initialized from the model checkpoint at google/pegasus-xsum and are newly initialized: ['model.decoder.embed_positions.weight', 'model.encoder.embed_positions.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


✅ Model ready.
▶️ [pegasus-xsum] Running test400, round 1 ...
✅ [pegasus-xsum] Done test400 run 1: 400 rows → ./pegasus_infer_schedule/pegasus-xsum/pred_test400_run1.csv
▶️ [pegasus-xsum] Running test400, round 2 ...
✅ [pegasus-xsum] Done test400 run 2: 400 rows → ./pegasus_infer_schedule/pegasus-xsum/pred_test400_run2.csv


In [11]:
!zip -r -q /kaggle/working//pegasus_xsum_infer.zip /kaggle/working/

huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)


# BART

In [5]:
# -*- coding: utf-8 -*-
# Inference-only với BART (facebook/bart-large)
# - Không HPO, không fine-tune, không LoRA
# - Không hậu kỳ cắt theo ký tự
# - Thêm tham số generate để loại warning "max_length=20"
import os, time, csv
import torch
import pandas as pd
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

# ----------- CONFIG -----------
TEST_CSV = "/kaggle/input/project-multi/test_random_400-2.csv"
MODEL_NAME = "facebook/bart-large"      # base, chưa fine-tune
PRED_CSV = "bart_predictions_zero_shot.csv"
INFER_LOG_CSV = "inference_logs_zero_shot.csv"
MAX_SOURCE_LEN = 1000                   # giới hạn input theo token (encoder)
# Tham số generate khuyến nghị (có thể đổi tùy ý)
GEN_KW = dict(
    max_new_tokens=64,                  # kiểm soát độ dài output, tránh default max_length=20
    num_beams=4,                        # beam search cho đầu ra ổn định hơn
    no_repeat_ngram_size=3,             # hạn chế lặp
    length_penalty=2.0,                 # khuyến khích câu ngắn gọn hơn
    early_stopping=True,
    do_sample=False,                    # deterministic
)

# ----------- Load data -----------
assert os.path.exists(TEST_CSV), f"Missing: {TEST_CSV}"
df = pd.read_csv(TEST_CSV)

TEST_INPUT_COL = None
for cand in ["description_html_clean", "description_html"]:
    if cand in df.columns:
        TEST_INPUT_COL = cand
        break
assert TEST_INPUT_COL is not None, "Test CSV cần 'description_html_clean' hoặc 'description_html'"

print("⏳ Loading base model...")
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, use_fast=True)
model = AutoModelForSeq2SeqLM.from_pretrained(MODEL_NAME)
model.eval()
device = "cuda" if torch.cuda.is_available() else "cpu"
model.to(device)
print("✅ Model ready on", device)

# chuẩn bị file log
with open(INFER_LOG_CSV, "w", newline="", encoding="utf-8") as f:
    csv.writer(f).writerow(["index","input_tokens","gen_tokens","latency_sec"])

@torch.inference_mode()
def infer_one(html: str, idx: int) -> str:
    # Không dùng prompt đặc thù; đưa thẳng văn bản vào encoder
    src = str(html)
    enc = tokenizer(
        src,
        return_tensors="pt",
        truncation=True,
        max_length=MAX_SOURCE_LEN,
    )
    enc = {k: v.to(device) for k, v in enc.items()}

    t0 = time.time()
    out = model.generate(**enc, **GEN_KW)
    dur = time.time() - t0

    # log
    input_tokens = int(enc["attention_mask"].sum().item())
    gen_tokens = int(out.shape[1] - enc["input_ids"].shape[1])
    with open(INFER_LOG_CSV, "a", newline="", encoding="utf-8") as f:
        csv.writer(f).writerow([idx, input_tokens, gen_tokens, f"{dur:.4f}"])

    text = tokenizer.decode(out[0], skip_special_tokens=True, clean_up_tokenization_spaces=True).strip()
    return text

# ----------- Run inference -----------
preds = []
for i, html in enumerate(df[TEST_INPUT_COL].astype(str).tolist()):
    preds.append(infer_one(html, i))

df["bart_pred"] = preds
df.to_csv(PRED_CSV, index=False, encoding="utf-8")
print(f"🎯 Done. Saved predictions to: {PRED_CSV}")
print(f"🗒  Logs at: {INFER_LOG_CSV}")

⏳ Loading base model...


/usr/local/lib/python3.11/dist-packages/transformers/tokenization_utils_base.py:1601: FutureWarning: `clean_up_tokenization_spaces` was not set. It will be set to `True` by default. This behavior will be depracted in transformers v4.45, and will be then set to `False` by default. For more details check this issue: https://github.com/huggingface/transformers/issues/31884
  warnings.warn(


✅ Model ready on cuda
🎯 Done. Saved predictions to: bart_predictions_zero_shot.csv
🗒  Logs at: inference_logs_zero_shot.csv


In [7]:
# -*- coding: utf-8 -*-
# Inference-only BART (facebook/bart-large & bart-large-xsum)
# - Chạy xen kẽ theo lịch: 600 → 400 → 600 → 400 → 600
# - Không HPO, không fine-tune, không LoRA
# - Không hậu kỳ cắt theo ký tự
# - Giữ tham số generate như yêu cầu

import os, time, csv
import torch
import pandas as pd
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

# ===== CONFIG =====
INPUT_FILES = {
    "test400": "/kaggle/input/project-multi/test_random_400-2.csv",
    "test600": "/kaggle/input/project-multi/test_random_600-2.csv",
}
RUN_SCHEDULE = ["test400", "test600", "test400", "test600", "test400", "test600"]  # xen kẽ & đúng số lần
MODELS = {
    "bart-base": "facebook/bart-large",
    # "bart-xsum": "facebook/bart-large-xsum",
}

MAX_SOURCE_LEN = 1000
GEN_KW = dict(
    max_new_tokens=64,      # tránh cảnh báo max_length=20
    num_beams=4,
    no_repeat_ngram_size=3,
    length_penalty=1.0,
    early_stopping=True,
    do_sample=False,
)

INPUT_COL_CANDIDATES = ["description_html_clean", "description_html"]

# ===== RUNNER =====
def run_one_round(model_tag, model_name, test_tag, run_id):
    test_file = INPUT_FILES[test_tag]
    assert os.path.exists(test_file), f"Missing: {test_file}"
    df = pd.read_csv(test_file)

    test_input_col = None
    for c in INPUT_COL_CANDIDATES:
        if c in df.columns:
            test_input_col = c
            break
    assert test_input_col is not None, f"CSV {test_file} cần 1 trong các cột {INPUT_COL_CANDIDATES}"

    pred_csv = f"{model_tag}_{test_tag}_run{run_id}_pred.csv"
    log_csv  = f"{model_tag}_{test_tag}_run{run_id}_log.csv"

    print(f"\n⏳ Loading {model_name} | {test_tag} (run {run_id})")
    tokenizer = AutoTokenizer.from_pretrained(model_name, use_fast=True)
    model = AutoModelForSeq2SeqLM.from_pretrained(model_name)
    model.eval()
    device = "cuda" if torch.cuda.is_available() else "cpu"
    model.to(device)
    print(f"✅ Model ready on {device}")

    with open(log_csv, "w", newline="", encoding="utf-8") as f:
        csv.writer(f).writerow(["index", "input_tokens", "gen_tokens", "latency_sec"])

    @torch.inference_mode()
    def infer_one(text: str, idx: int) -> str:
        enc = tokenizer(
            str(text),
            return_tensors="pt",
            truncation=True,
            max_length=MAX_SOURCE_LEN,
        )
        enc = {k: v.to(device) for k, v in enc.items()}

        t0 = time.time()
        out = model.generate(**enc, **GEN_KW)
        dur = time.time() - t0

        # input tokens
        input_tokens = int(enc["attention_mask"].sum().item())

        # output tokens (seq2seq generate trả riêng decoder output)
        gen_tokens = int((out[0] != tokenizer.pad_token_id).sum().item())

        with open(log_csv, "a", newline="", encoding="utf-8") as f:
            csv.writer(f).writerow([idx, input_tokens, gen_tokens, f"{dur:.4f}"])

        return tokenizer.decode(out[0], skip_special_tokens=True, clean_up_tokenization_spaces=True).strip()

    preds = []
    for i, src in enumerate(df[test_input_col].astype(str).tolist()):
        preds.append(infer_one(src, i))

    df[f"{model_tag}_pred"] = preds
    df.to_csv(pred_csv, index=False, encoding="utf-8")
    print(f"🎯 Saved predictions → {pred_csv}")
    print(f"🗒  Logs → {log_csv}")

# ===== MAIN (interleaved schedule) =====
if __name__ == "__main__":
    for model_tag, model_name in MODELS.items():
        # đếm số lần theo từng file để đặt run_id đúng 1..N
        counters = {"test400": 0, "test600": 0}
        for test_tag in RUN_SCHEDULE:
            counters[test_tag] += 1
            run_one_round(model_tag, model_name, test_tag, counters[test_tag])


⏳ Loading facebook/bart-large | test400 (run 1)


/usr/local/lib/python3.11/dist-packages/transformers/tokenization_utils_base.py:1601: FutureWarning: `clean_up_tokenization_spaces` was not set. It will be set to `True` by default. This behavior will be depracted in transformers v4.45, and will be then set to `False` by default. For more details check this issue: https://github.com/huggingface/transformers/issues/31884
  warnings.warn(


✅ Model ready on cuda
🎯 Saved predictions → bart-base_test400_run1_pred.csv
🗒  Logs → bart-base_test400_run1_log.csv

⏳ Loading facebook/bart-large | test600 (run 1)


/usr/local/lib/python3.11/dist-packages/transformers/tokenization_utils_base.py:1601: FutureWarning: `clean_up_tokenization_spaces` was not set. It will be set to `True` by default. This behavior will be depracted in transformers v4.45, and will be then set to `False` by default. For more details check this issue: https://github.com/huggingface/transformers/issues/31884
  warnings.warn(


✅ Model ready on cuda
🎯 Saved predictions → bart-base_test600_run1_pred.csv
🗒  Logs → bart-base_test600_run1_log.csv

⏳ Loading facebook/bart-large | test400 (run 2)


...

✅ Model ready on cuda
🎯 Saved predictions → bart-base_test600_run2_pred.csv
🗒  Logs → bart-base_test600_run2_log.csv

⏳ Loading facebook/bart-large | test400 (run 3)


/usr/local/lib/python3.11/dist-packages/transformers/tokenization_utils_base.py:1601: FutureWarning: `clean_up_tokenization_spaces` was not set. It will be set to `True` by default. This behavior will be depracted in transformers v4.45, and will be then set to `False` by default. For more details check this issue: https://github.com/huggingface/transformers/issues/31884
  warnings.warn(


✅ Model ready on cuda
🎯 Saved predictions → bart-base_test400_run3_pred.csv
🗒  Logs → bart-base_test400_run3_log.csv

⏳ Loading facebook/bart-large | test600 (run 3)


/usr/local/lib/python3.11/dist-packages/transformers/tokenization_utils_base.py:1601: FutureWarning: `clean_up_tokenization_spaces` was not set. It will be set to `True` by default. This behavior will be depracted in transformers v4.45, and will be then set to `False` by default. For more details check this issue: https://github.com/huggingface/transformers/issues/31884
  warnings.warn(


✅ Model ready on cuda
🎯 Saved predictions → bart-base_test600_run3_pred.csv
🗒  Logs → bart-base_test600_run3_log.csv


# Gemma

In [ ]:
!pip install -q -U transformers accelerate datasets peft bitsandbytes

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.1/40.1 kB 2.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.6/11.6 MB 90.3 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 374.9/374.9 kB 21.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 503.6/503.6 kB 14.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 504.9/504.9 kB 23.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.1/60.1 MB 27.0 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 564.3/564.3 kB 28.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 42.8/42.8 MB 39.7 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 69.9 MB/s eta 0:00:00:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.4/363.4 MB 4.6 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.8/13.8 MB 79.9 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.6/24.6 MB 69.9 

In [4]:
# -*- coding: utf-8 -*-
# Inference-only với Gemma-2-2b-it
# - Không fine-tune, không LoRA
# - Input: test CSV (description_html_clean hoặc description_html)
# - Output: pred_short_description

import os, time, csv
import torch
import pandas as pd
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

# -------- CONFIG --------
TEST_CSV = "/kaggle/input/project-multi/test_random_400-2.csv"
MODEL_NAME = "google/"   # model gốc
OUT_CSV = "gemma_pretrain_predictions.csv"
INFER_LOG_CSV = "inference_logs_gemma_pretrain.csv"

MAX_SOURCE_LEN = 2048
MAX_NEW_TOKENS = 64

# -------- Load data --------
assert os.path.exists(TEST_CSV), f"Missing: {TEST_CSV}"
df = pd.read_csv(TEST_CSV)

TEST_INPUT_COL = None
for cand in ["description_html_clean", "description_html"]:
    if cand in df.columns:
        TEST_INPUT_COL = cand
        break
assert TEST_INPUT_COL is not None, "CSV cần cột 'description_html_clean' hoặc 'description_html'"

def build_prompt(description_html: str) -> str:
    return (
        "You are an expert app store editor. "
        "Given the following app description in HTML format, summarize it in 2-3 sentences, "
        "with a concise, engaging short description (max 80 characters) suitable for an app store listing. "
        f"App Description HTML:\n{description_html}\n"
        "Format your response as:\n"
        "Short Description: <your short description>\n\n"
    )

# -------- Load model --------
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
)

print("⏳ Loading model...")
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, use_fast=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    quantization_config=bnb_config,
    torch_dtype=torch.float16,
    device_map="auto",
)
model.eval()
print("✅ Model ready.")

# -------- Prepare log --------
with open(INFER_LOG_CSV, "w", newline="", encoding="utf-8") as f:
    csv.writer(f).writerow(["index","input_tokens","gen_tokens","latency_sec"])

@torch.inference_mode()
def generate_one(html: str, idx: int) -> str:
    src = build_prompt(html)
    enc = tokenizer(src, return_tensors="pt", truncation=True, max_length=MAX_SOURCE_LEN)
    enc = {k: v.to(model.device) for k, v in enc.items()}

    t0 = time.time()
    out = model.generate(
        **enc,
        max_new_tokens=MAX_NEW_TOKENS,
        do_sample=False,
        temperature=0.0,
        top_p=1.0,
        eos_token_id=tokenizer.eos_token_id,
    )
    dur = time.time() - t0

    input_tokens = int(enc["input_ids"].shape[1])
    gen_tokens = int(out.shape[1] - enc["input_ids"].shape[1])
    with open(INFER_LOG_CSV, "a", newline="", encoding="utf-8") as f:
        csv.writer(f).writerow([idx, input_tokens, gen_tokens, f"{dur:.4f}"])

    gen = out[0][enc["input_ids"].shape[1]:]
    return tokenizer.decode(gen, skip_special_tokens=True).strip()

# -------- Run inference --------
preds = []
for i, html in enumerate(df[TEST_INPUT_COL].astype(str).tolist()):
    preds.append(generate_one(html, i))

df["pred_short_description"] = preds
df.to_csv(OUT_CSV, index=False, encoding="utf-8")
print(f"🎯 Done. Predictions saved to: {OUT_CSV}")
print(f"🗒 Logs saved to: {INFER_LOG_CSV}")

⏳ Loading model...


tokenizer_config.json:   0%|          | 0.00/47.0k [00:00<?, ?B/s]

tokenizer.model:   0%|          | 0.00/4.24M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/17.5M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/636 [00:00<?, ?B/s]

...

model-00002-of-00002.safetensors:   0%|          | 0.00/241M [00:00<?, ?B/s]

Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/187 [00:00<?, ?B/s]

The following generation flags are not valid and may be ignored: ['temperature']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


✅ Model ready.
🎯 Done. Predictions saved to: gemma_pretrain_predictions.csv
🗒 Logs saved to: inference_logs_gemma_pretrain.csv


In [ ]:
# -*- coding: utf-8 -*-
# Inference-only với T5 (encoder-decoder)
# - Không fine-tune, không LoRA
# - Input: test CSV (description_html_clean hoặc description_html)
# - Output: pred_short_description (kèm full_summary tùy chọn)

import os, time, csv, re
import torch
import pandas as pd
from transformers import AutoModelForSeq2SeqLM, AutoTokenizer, BitsAndBytesConfig

# -------- CONFIG --------
TEST_CSV = "/kaggle/input/project-multi/test_random_400-2.csv"
MODEL_NAME = "t5-3b"  # hoặc "google/t5-v1_1-xxl" (nếu bạn muốn bản v1.1); giữ "google-t5/t5-3b" nếu đó là repo bạn dùng
OUT_CSV = "t5_predictions.csv"
INFER_LOG_CSV = "inference_logs_t5.csv"

# Lưu ý: T5 gốc thường train với input length ~512. Đặt 512–768 để an toàn.
MAX_SOURCE_LEN = 512
MAX_NEW_TOKENS = 48   # tổng độ dài summary sinh ra
BATCH_SIZE = 2          # điều chỉnh theo VRAM
GEN_KW = dict(
    do_sample=False,
    num_beams=4,
    length_penalty=1.0,
    max_new_tokens=MAX_NEW_TOKENS,
    no_repeat_ngram_size=3,
    early_stopping=True,
)

# -------- Load data --------
assert os.path.exists(TEST_CSV), f"Missing: {TEST_CSV}"
df = pd.read_csv(TEST_CSV)

TEST_INPUT_COL = None
for cand in ["description_html_clean", "description_html"]:
    if cand in df.columns:
        TEST_INPUT_COL = cand
        break
assert TEST_INPUT_COL is not None, "CSV cần cột 'description_html_clean' hoặc 'description_html'"

def build_t5_input(description_html: str) -> str:
    # T5 dùng tiền tố "summarize: " để biết tác vụ
    # Bạn có thể ràng buộc format đầu ra bằng chỉ dẫn ngắn gọn.
    instr = (
        "Summarize the following app description (HTML) in 2-3 sentences. "
        "Then produce a short, catchy tagline (<= 80 characters) suitable for an app store.\n"
        "Output format:\n"
        "Short Description: <your short description>\n"
        "Summary: <2-3 sentences>\n"
        "TEXT:\n"
    )
    return f"summarize: {instr}{description_html}"

# -------- Load model --------
use_4bit = True  # tắt nếu không có bitsandbytes
if use_4bit:
    bnb_config = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_use_double_quant=True,
        bnb_4bit_quant_type="nf4",
        bnb_4bit_compute_dtype=torch.float16,
    )
else:
    bnb_config = None

print("⏳ Loading model...")
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, use_fast=True)
# T5 không có EOS như GPT; pad token đã có sẵn, nhưng đảm bảo không None
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForSeq2SeqLM.from_pretrained(
    MODEL_NAME,
    quantization_config=bnb_config,
    torch_dtype=torch.float16 if use_4bit else torch.float32,
    device_map="auto",
)
model.eval()
print("✅ Model ready.")

# -------- Prepare log --------
with open(INFER_LOG_CSV, "w", newline="", encoding="utf-8") as f:
    csv.writer(f).writerow(["index", "input_tokens", "gen_tokens", "latency_sec"])

# -------- Helper: parse output --------
SHORT_RE = re.compile(r"^Short Description:\s*(.+)", flags=re.IGNORECASE)
SUM_RE = re.compile(r"^Summary:\s*(.+)", flags=re.IGNORECASE)

def parse_outputs(text: str):
    short_desc, full_sum = None, None
    lines = [ln.strip() for ln in text.splitlines() if ln.strip()]
    for ln in lines:
        m1 = SHORT_RE.match(ln)
        if m1 and short_desc is None:
            short_desc = m1.group(1).strip()
        m2 = SUM_RE.match(ln)
        if m2 and full_sum is None:
            full_sum = m2.group(1).strip()
    # fallback nếu model không giữ format
    if short_desc is None:
        # lấy câu đầu/chuỗi ngắn nhất làm short desc dự phòng
        short_desc = (lines[0] if lines else text)[:80]
    if full_sum is None:
        full_sum = text
    # cắt ngắn short_desc về <= 80 ký tự
    if len(short_desc) > 80:
        short_desc = short_desc[:77].rstrip() + "..."
    return short_desc, full_sum

# -------- Inference loop (batched) --------
pred_rows = []
inputs = df[TEST_INPUT_COL].astype(str).tolist()

def batch_iter(lst, bs):
    for i in range(0, len(lst), bs):
        yield i, lst[i:i+bs]

for start_idx, chunk in batch_iter(inputs, BATCH_SIZE):
    prompts = [build_t5_input(x) for x in chunk]
    enc = tokenizer(
        prompts,
        max_length=MAX_SOURCE_LEN,
        padding=True,
        truncation=True,
        return_tensors="pt",
    )
    enc = {k: v.to(model.device) for k, v in enc.items()}
    torch.cuda.empty_cache()
    t0 = time.time()
    with torch.inference_mode():
        out = model.generate(
            **enc,
            **GEN_KW,
        )
    latency = time.time() - t0

    decoded = tokenizer.batch_decode(out, skip_special_tokens=True)
    # logging + save rows
    input_lens = enc["input_ids"].ne(tokenizer.pad_token_id).sum(dim=1).tolist()
    gen_lens = out.ne(tokenizer.pad_token_id).sum(dim=1).tolist()

    # ghi log từng item trong batch
    with open(INFER_LOG_CSV, "a", newline="", encoding="utf-8") as f:
        w = csv.writer(f)
        for j, (inp_len, gen_len) in enumerate(zip(input_lens, gen_lens)):
            w.writerow([start_idx + j, inp_len, gen_len, round(latency, 4)])

    for j, text in enumerate(decoded):
        short_desc, full_sum = parse_outputs(text)
        pred_rows.append({
            "index": start_idx + j,
            "short_description": short_desc,
            "full_summary": full_sum,
        })

# -------- Write outputs --------
pred_df = pd.DataFrame(pred_rows).sort_values("index").reset_index(drop=True)
pred_df.to_csv(OUT_CSV, index=False, encoding="utf-8")
print(f"✅ Done. Wrote {len(pred_df)} rows to: {OUT_CSV}")
print(f"📝 Logs: {INFER_LOG_CSV}")